In [1]:
# Parameters
DB_PATH                   = "../../../DB/oedb_refiner_1st.db"
BENCHMARK_PATH            = "../../../data/input_data/benchmark_trainingset.xlsx"
BENCHMARK_SHEET           = "merged_answers"
MATCHED_PARTICIPANTS_CSV  = "matched_participants.csv"
MATCHED_QUESTIONS_CSV     = "matched_questions.csv"
MATCHED_CSV_PATH          = "matched_answers.csv"
NOTEGROUP_ID_MIN          = 1
NOTEGROUP_ID_MAX          = 23
MATCH_THRESHOLD           = 75   # score scale is 0-100

WEIGHT_CONTENT     = 0.8
WEIGHT_PARTICIPANT = 0.1
WEIGHT_QUESTION     = 0.1

In [2]:
import sqlite3
import re
import pandas as pd
from rapidfuzz import fuzz

def load_etl(db_path, id_min, id_max):
    con = sqlite3.connect(db_path)
    df = pd.read_sql_query(
        """SELECT answerID, notegroupID, questionID, participantID,
                  answer_content_oriLAN, answer_content_EN
           FROM answers
           WHERE notegroupID BETWEEN ? AND ?""",
        con, params=(id_min, id_max)
    )
    con.close()
    df["answerID"] = df["answerID"].astype(int)
    return df.set_index("answerID")

def load_benchmark(xlsx_path, sheet, id_min, id_max):
    df = pd.read_excel(xlsx_path, sheet_name=sheet, dtype=str)
    df["notegroupID"] = df["notegroupID"].astype(int)
    df["answerID"]    = df["answerID"].astype(int)
    df = df[df["notegroupID"].between(id_min, id_max)]
    return df.set_index("answerID")

# Load question_content lookup tables for question_match fallback
def load_etl_questions(db_path, id_min, id_max):
    con = sqlite3.connect(db_path)
    df = pd.read_sql_query(
        "SELECT questionID, question_content FROM questions WHERE notegroupID BETWEEN ? AND ?",
        con, params=(id_min, id_max)
    )
    con.close()
    df["questionID"] = df["questionID"].astype(int)
    return df.set_index("questionID")

def load_bm_questions(xlsx_path, id_min, id_max):
    df = pd.read_excel(xlsx_path, sheet_name="questions", dtype=str)
    df = df.dropna(subset=["notegroupID"])
    df["notegroupID"] = df["notegroupID"].astype(int)
    df["questionID"]  = df["questionID"].astype(int)
    df = df[df["notegroupID"].between(id_min, id_max)]
    return df.set_index("questionID")

etl_questions = load_etl_questions(DB_PATH, NOTEGROUP_ID_MIN, NOTEGROUP_ID_MAX)
bm_questions  = load_bm_questions(BENCHMARK_PATH, NOTEGROUP_ID_MIN, NOTEGROUP_ID_MAX)

def load_id_map(csv_path, etl_col, bm_col):
    """Load a matched-pairs CSV and return etl->bm and bm->etl id dicts."""
    df = pd.read_csv(csv_path)
    df[etl_col] = df[etl_col].astype(int)
    df[bm_col]  = df[bm_col].astype(int)
    etl_to_bm = dict(zip(df[etl_col], df[bm_col]))
    bm_to_etl = dict(zip(df[bm_col], df[etl_col]))
    return etl_to_bm, bm_to_etl

etl = load_etl(DB_PATH, NOTEGROUP_ID_MIN, NOTEGROUP_ID_MAX)
bm  = load_benchmark(BENCHMARK_PATH, BENCHMARK_SHEET, NOTEGROUP_ID_MIN, NOTEGROUP_ID_MAX)

participant_etl_to_bm, participant_bm_to_etl = load_id_map(
    MATCHED_PARTICIPANTS_CSV, "etl_participantID", "bm_participantID"
)
question_etl_to_bm, question_bm_to_etl = load_id_map(
    MATCHED_QUESTIONS_CSV, "etl_questionID", "bm_questionID"
)

print("ETL records:      ", len(etl))
print("Benchmark records:", len(bm))
print("Matched participants:", len(participant_etl_to_bm))
print("Matched questions:   ", len(question_etl_to_bm))

ETL records:       1218
Benchmark records: 1232
Matched participants: 87
Matched questions:    415


In [3]:
def normalise_str(val):
    if pd.isna(val) or str(val).strip() in ("", "None", "nan"):
        return None
    s = str(val).strip().lower()
    s = re.sub(r'\s*\n\s*', '\n', s)
    return s

def normalise_id(val):
    """Normalise a participantID/questionID value to int, or None."""
    if pd.isna(val) or str(val).strip() in ("", "None", "nan"):
        return None
    try:
        return int(float(str(val).strip()))
    except (ValueError, TypeError):
        return None

def participant_match(etl_row, bm_row):
    """
    Return 1 if ETL participantID, mapped to BM space, equals BM row's participantID,
    or if both sides are null. Else 0.
    """
    etl_pid = normalise_id(etl_row.get("participantID"))
    bm_pid  = normalise_id(bm_row.get("participantID"))
    if etl_pid is None and bm_pid is None:
        return 1
    mapped_pid = participant_etl_to_bm.get(etl_pid) if etl_pid is not None else None
    return 1 if mapped_pid is not None and mapped_pid == bm_pid else 0

def question_match(etl_row, bm_row):
    """
    Return 1 if ETL questionID, mapped to BM space, equals BM row's questionID,
    OR if the question_content of both sides match via partial_ratio >= 0.85,
    or if both sides are null. Else 0.
    """
    etl_qid = normalise_id(etl_row.get("questionID"))
    bm_qid  = normalise_id(bm_row.get("questionID"))
    if etl_qid is None and bm_qid is None:
        return 1
    # Try ID mapping
    mapped_qid = question_etl_to_bm.get(etl_qid) if etl_qid is not None else None
    if mapped_qid is not None and mapped_qid == bm_qid:
        return 1
    # Fallback: compare question_content via partial_ratio
    etl_qc = normalise_str(etl_questions.at[etl_qid, "question_content"]) \
             if etl_qid is not None and etl_qid in etl_questions.index else None
    bm_qc  = normalise_str(bm_questions.at[bm_qid, "question_content"]) \
             if bm_qid  is not None and bm_qid  in bm_questions.index  else None
    if etl_qc is not None and bm_qc is not None:
        if fuzz.partial_ratio(etl_qc, bm_qc) >= 85:
            return 1
    return 0

def safe_ratio(a, b):
    if a is None or b is None:
        return None
    return fuzz.ratio(a, b)

def content_score(etl_row, bm_row):
    """
    Compare answer_content_oriLAN / answer_content_EN, allowing for the
    possibility that the two fields are cross-matched (swapped).
    Returns the highest single-field similarity score across all four
    possible pairings (0-100). Null pairs are excluded.
    """
    etl_ori = normalise_str(etl_row.get("answer_content_oriLAN"))
    etl_en  = normalise_str(etl_row.get("answer_content_EN"))
    bm_ori  = normalise_str(bm_row.get("answer_content_oriLAN"))
    bm_en   = normalise_str(bm_row.get("answer_content_EN"))

    scores = [
        safe_ratio(etl_ori, bm_ori),
        safe_ratio(etl_en,  bm_en),
        safe_ratio(etl_ori, bm_en),
        safe_ratio(etl_en,  bm_ori),
    ]
    scores = [s for s in scores if s is not None]

    return max(scores) if scores else 0.0

def answer_pair_score(etl_row, bm_row):
    p_score = participant_match(etl_row, bm_row) * 100
    q_score = question_match(etl_row, bm_row) * 100
    c_score = content_score(etl_row, bm_row)
    return WEIGHT_PARTICIPANT * p_score + WEIGHT_QUESTION * q_score + WEIGHT_CONTENT * c_score

def is_substring_match(etl_row, bm_row, threshold):
    """
    Secondary match condition: BM answer_content_oriLAN closely aligns with
    a portion of the ETL text (handles merged/interrupted answers), AND
    supporting fields (participant_match, question_match) both equal 1.
    """
    e = normalise_str(etl_row.get("answer_content_oriLAN"))
    b = normalise_str(bm_row.get("answer_content_oriLAN"))
    if e is None or b is None:
        return False

    # partial_ratio finds the best-aligned substring match, tolerant of
    # interruptions like inserted speaker tags
    if fuzz.partial_ratio(e, b) < threshold:
        return False

    p_match = participant_match(etl_row, bm_row)
    q_match = question_match(etl_row, bm_row)

    return p_match == 1 and q_match == 1

In [4]:
def map_answers(etl_df, bm_df, threshold):
    """
    Match answers within each notegroupID by weighted similarity:
    80% content similarity (direct or cross oriLAN/EN alignment),
    10% participant identity match, 10% question identity match (via prior mappings).
    Returns:
        matched  : list of (etl_idx, bm_idx, score)
        etl_only : list of etl_idx  -> FP rows
        bm_only  : list of bm_idx   -> FN rows
    """
    matched  = []
    etl_only = []
    bm_only  = []
    
    all_ng_ids = sorted(set(etl_df["notegroupID"].unique()) | set(bm_df["notegroupID"].unique()))
    for ng_id in all_ng_ids:
        etl_ng = etl_df[etl_df["notegroupID"] == ng_id]
        bm_ng  = bm_df[bm_df["notegroupID"]  == ng_id]

        if bm_ng.empty:
            etl_only.extend(etl_ng.index.tolist())
            continue
        if etl_ng.empty:
            bm_only.extend(bm_ng.index.tolist())
            continue

        scores = {}
        for ei in etl_ng.index:
            for bi in bm_ng.index:
                primary_score = answer_pair_score(etl_ng.loc[ei], bm_ng.loc[bi])
                if primary_score < threshold and is_substring_match(etl_ng.loc[ei], bm_ng.loc[bi], 85):
                    primary_score = threshold
                scores[(ei, bi)] = primary_score

        used_etl = set()
        used_bm  = set()
        for (ei, bi), score in sorted(scores.items(), key=lambda x: -x[1]):
            if score < threshold:
                break
            if ei in used_etl or bi in used_bm:
                continue
            matched.append((ei, bi, round(score, 2)))
            used_etl.add(ei)
            used_bm.add(bi)

        etl_only.extend([i for i in etl_ng.index if i not in used_etl])
        bm_only.extend( [i for i in bm_ng.index  if i not in used_bm])

    return matched, etl_only, bm_only


matched, etl_only, bm_only = map_answers(etl, bm, MATCH_THRESHOLD)

print(f"Matched pairs : {len(matched)}")
print(f"ETL-only (FP) : {len(etl_only)}")
print(f"BM-only  (FN) : {len(bm_only)}")

Matched pairs : 1204
ETL-only (FP) : 14
BM-only  (FN) : 28


In [5]:
print("=== ETL-only (FP) ===")
display(etl.loc[etl_only, ["notegroupID", "participantID", "questionID", "answer_content_oriLAN", "answer_content_EN"]])

print("\n=== BM-only (FN) ===")
bm_cols = [c for c in ["notegroupID", "participantID", "questionID", "answer_content_oriLAN", "answer_content_EN"] if c in bm.columns]
display(bm.loc[bm_only, bm_cols])

=== ETL-only (FP) ===


,notegroupID,participantID,questionID,answer_content_oriLAN,answer_content_EN
answerID,,,,,
273,4,39.0,61,"No, I’m not searching for work at all at the m...",NaN
436,6,42.0,129,What is needed to improve this? (examples: mot...,NaN
594,9,52.0,187,-,
598,9,52.0,188,-,
609,9,52.0,194,-,
815,12,64.0,268,1.) Inform them: how to get in touch with the ...,
926,18,81.0,318,When I was in the camp I worked. But now I do ...,NaN
1053,21,92.0,353,"(Vrouw, 23, B1)",NaN
1054,21,93.0,353,"(Vrouw, 42, B1)",NaN



=== BM-only (FN) ===


,notegroupID,participantID,questionID,answer_content_oriLAN,answer_content_EN
answerID,,,,,
189,3,24,40,"they taught us, Dutch language. We took certif...",NaN
250,4,NaN,55,We answered it before,NaN
1228,5,34,106,Racist people here.,NaN
1229,11,51,261,"Den Helder met afstand naar ons.""\n""Ik mis mij...",NaN
826,12,NaN,280,Inform them: how to get in touch with the muni...,NaN
827,12,NaN,280,Inform who you need to be in contact with,NaN
828,12,NaN,280,Which questions can I ask to the municipality?,NaN
844,13,58,290,online by email or survey,NaN
845,13,61,290,sessions and personal talks from the municipality,NaN


In [6]:
csv_rows = []
for ei, bi, score in matched:
    csv_rows.append({
        "notegroupID":   etl.loc[ei, "notegroupID"],
        "etl_answerID":  ei,
        "bm_answerID":   bi,
        "etl_oriLAN":    etl.loc[ei, "answer_content_oriLAN"],
        "bm_oriLAN":     bm.loc[bi, "answer_content_oriLAN"] if "answer_content_oriLAN" in bm.columns else None,
        "etl_EN":        etl.loc[ei, "answer_content_EN"],
        "bm_EN":         bm.loc[bi, "answer_content_EN"] if "answer_content_EN" in bm.columns else None,
        "match_score":   score,
    })

matched_csv = pd.DataFrame(csv_rows)
matched_csv.to_csv(MATCHED_CSV_PATH, index=False)
print(f"Saved {len(matched_csv)} matched pairs to {MATCHED_CSV_PATH}")

Saved 1204 matched pairs to matched_answers.csv
